# MCTS Training for Catanbot (Colab)

This notebook is designed for Google Colab. It mounts Google Drive, installs dependencies, imports the `mcts` package from this repository, and runs the MCTS trade model training pipeline.

In [ ]:
from google.colab import drive
import os
import sys

# Mount your Google Drive and set the repo root path.
drive.mount('/content/drive')

# TODO: update this path to your Catanbot repository folder in Drive.
repo_path = '/content/drive/MyDrive/Catanbot'
print('Repo path:', repo_path)
os.chdir(repo_path)
print('Working directory:', os.getcwd())

# Add repository to sys.path for imports.
sys.path.append(repo_path)


In [ ]:
# Verify that the code on Google Drive is up to date with necessary fixes.
# Check that enums has the LogType attributes we need.
import subprocess

print("Checking if enums.py has LogType.RESOURCE_RECEIVED...")
result = subprocess.run(
    ['grep', '-c', 'RESOURCE_RECEIVED', 'data/enums.py'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✓ enums.py is up to date')
else:
    print('⚠ WARNING: enums.py may be outdated on Drive')
    print('  Make sure you synced the latest code from GitHub to Google Drive')


In [ ]:
# Install any required packages.
# Colab typically already has torch and numpy, but we install extras from requirements.
!pip install -r requirements.txt

# Optionally, if this Colab runtime has a GPU and you want to use it:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [ ]:
# Import the MCTS modules from the repository.
# Force reload to ensure latest code is used (important after code edits).
import sys
import importlib

# Remove old module references completely
mods_to_remove = [m for m in sys.modules.keys() if m.startswith('mcts') or m.startswith('data')]
for mod in mods_to_remove:
    del sys.modules[mod]

# Now import fresh
from mcts.run_pipeline import validate_hand_tracker, extract_training_data, train_models, run_search
from mcts.trade_models import TradeAcceptanceModel, TradeProposalPolicy
from mcts.hand_tracker import HandTracker
from mcts.trade_encoder import TradeEncoder
from mcts.search import TradeMCTS, find_best_trade
from data.enums import LogType

# Verify LogType has the expected attributes
print('mcts module imported successfully (reloaded)')
print(f'LogType attributes available: {len([a for a in dir(LogType) if not a.startswith("_")])} enum values')

# Quick check
if hasattr(LogType, 'RESOURCE_RECEIVED'):
    print('✓ LogType.RESOURCE_RECEIVED is available')
else:
    print('⚠ WARNING: LogType.RESOURCE_RECEIVED not found')
    print('Available LogType attributes:', [a for a in dir(LogType) if not a.startswith("_")][:10])


In [ ]:
# Configure paths and hyperparameters for training.
repo_root = os.getcwd()

games_dir = os.path.join(repo_root, 'dataset')
trade_data_dir = os.path.join(repo_root, 'trade_data')
trade_model_dir = os.path.join(repo_root, 'trade_models')

# Training hyperparameters.
epochs = 30
batch_size = 4096
use_gpu = False

# Use a smaller subset of the raw dataset to speed up Colab training.
# The repo contains ~44k game files, so 1/4 is roughly 11k games.
max_games = 11000

device = 'cuda' if use_gpu else 'cpu'

print('games_dir =', games_dir)
print('trade_data_dir =', trade_data_dir)
print('trade_model_dir =', trade_model_dir)
print('max_games =', max_games)
print('device =', device)


In [ ]:
# Initialize MCTS training helpers and verify data.
# This cell creates objects and prints status for the training pipeline.

encoder = TradeEncoder()
print('TradeEncoder total feature size:', encoder.total_feature_size)

# Check if dataset files exist.
print('Existing dataset files:')
for fname in ['acceptance_data.npz', 'proposal_data.npz']:
    path = os.path.join(trade_data_dir, fname)
    print(f'  {fname}:', os.path.exists(path))


In [ ]:
# If acceptance/proposal data files are missing, extract them from raw game logs.
accept_path = os.path.join(trade_data_dir, 'acceptance_data.npz')
proposal_path = os.path.join(trade_data_dir, 'proposal_data.npz')

if not os.path.exists(accept_path) or not os.path.exists(proposal_path):
    print('Training data missing, extracting from raw dataset...')
    extract_training_data(
        games_dir,
        output_dir=trade_data_dir,
        max_games=max_games,
        num_workers=4,
    )
else:
    print('Trade training data already exists, skipping extraction.')


In [ ]:
# Run model training for the trade acceptance and proposal models.
# If the data files are not yet generated, run the extract phase first.

train_models(
    data_dir=trade_data_dir,
    model_dir=trade_model_dir,
    epochs=epochs,
    device=device,
)


In [ ]:
# Verify saved model artifacts in Google Drive.
for fname in ['acceptance_model.pt', 'proposal_policy.pt', 'acceptance_history.npz']:
    path = os.path.join(trade_model_dir, fname)
    print(f'  {fname}:', os.path.exists(path), path)


In [ ]:
# Run a demonstration of MCTS on a single game state.
# This will use the saved trade models and print the recommended trade.

# Find a sample game file.
game_files = sorted([f for f in os.listdir(games_dir) if f.endswith('.json')])
if not game_files:
    raise FileNotFoundError('No JSON game files found in dataset directory')

sample_game_file = os.path.join(games_dir, game_files[0])
print('Sample game file:', sample_game_file)

best_trade = run_search(
    game_file=sample_game_file,
    turn=30,
    perspective=0,
    model_dir=trade_model_dir,
    iterations=1000,
)

print('Best trade recommendation:', best_trade)


## How MCTS integrates with Catan gameplay

The MCTS trade system is a separate decision module. It is meant to be called during a trade phase with the current game state, the observing player's perspective, and the hand-belief tracker.

It does not replace the main game model directly. Instead, it evaluates candidate trades and returns the best trade based on:
- candidate trade generation from the current state
- opponent hand uncertainty via `HandTracker`
- trade acceptance probability from `TradeAcceptanceModel`
- optional candidate pruning from `TradeProposalPolicy`
- leaf evaluation via a value function

So yes — when assembling the final Catan-playing agent, you can invoke this MCTS module only during trading decisions. It takes the known public state plus inferred beliefs and returns the best trade recommendation for that trading turn.

## Load trained models for inference

In [ ]:
# Load the trained acceptance model and proposal policy from disk.
acceptance_model = None
proposal_policy = None

accept_model_path = os.path.join(trade_model_dir, 'acceptance_model.pt')
if os.path.exists(accept_model_path):
    encoder = TradeEncoder()
    acceptance_model = TradeAcceptanceModel(encoder.total_feature_size)
    acceptance_model.load_state_dict(torch.load(accept_model_path, weights_only=True))
    acceptance_model.eval()
    print('✓ Loaded TradeAcceptanceModel')
else:
    print('⚠ TradeAcceptanceModel not found')

proposal_model_path = os.path.join(trade_model_dir, 'proposal_policy.pt')
if os.path.exists(proposal_model_path):
    encoder = TradeEncoder()
    proposal_policy = TradeProposalPolicy(encoder.total_feature_size)
    proposal_policy.load_state_dict(torch.load(proposal_model_path, weights_only=True))
    proposal_policy.eval()
    print('✓ Loaded TradeProposalPolicy')
else:
    print('⚠ TradeProposalPolicy not found')

print('\nModels ready for inference.')


In [ ]:
# Run MCTS trade search on a game state using the loaded models.
# This shows how to integrate the MCTS module into your agent.

# Sample game and parameters
game_files = sorted([f for f in os.listdir(games_dir) if f.endswith('.json')])
if game_files:
    sample_game = os.path.join(games_dir, game_files[0])
    print(f'Running MCTS on: {game_files[0]}')
    print(f'Turn: 30, Player perspective: 0\n')
    
    best_trade = run_search(
        game_file=sample_game,
        turn=30,
        perspective=0,
        model_dir=trade_model_dir,
        iterations=500,  # Use fewer for quick demo
    )
    
    if best_trade:
        print(f'MCTS recommends: {best_trade}')
    else:
        print('MCTS recommends: No trade')
else:
    print('⚠ No game files found in dataset directory')


### Troubleshooting module import issues

If you get an `AttributeError: type object 'LogType' has no attribute 'RESOURCE_RECEIVED'` error:

1. **Restart the kernel** — go to Runtime → Restart session in Colab
2. Re-run cells 1-4 in order (mount, install, reload modules, configure paths)
3. Then re-run the MCTS search cell

Colab sometimes caches old module versions. A fresh kernel restart ensures the latest code from Google Drive is used.